## Practice with Facebook Prophet

###  Installation in Python

In [2]:
# %pip install prophet

Let’s import the modules that we will need, and initialize our environment:

In [3]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np 
import pandas as pd
import statsmodels.api as sm
from scipy import stats

import matplotlib.pyplot as plt
#%config InlineBackend.figure_format = 'retina'

### Dataset

In [4]:
def get_file(filename, out_path:Path, overwrite=False):
    file_exists = (out_path/filename).exists()

    if file_exists:
        print("file exists")
    else:
        print("file not found")

    

In [5]:
FILE_NAME = "medium_posts.csv"
DATA_PATH =  Path(".")

get_file(filename=FILE_NAME, out_path=DATA_PATH)

file exists


In [6]:
df = pd.read_csv(DATA_PATH / FILE_NAME)
df.head()

,published,domain,url
0,2012-08-13 22:54:53.510Z,medium.com,https://medium.com/policy/medium-terms-of-serv...
1,2012-08-13 22:57:17.248Z,medium.com,https://medium.com/policy/medium-privacy-polic...
2,2016-11-04 23:40:43.364Z,medium.com,https://medium.com/@Medium/personalize-your-me...
3,2016-12-24 18:21:13.427Z,medium.com,https://medium.com/holiday-poems/xmas-morning-...
4,2015-09-22 21:37:48.207Z,blog.medium.com,https://blog.medium.com/taking-a-side-on-net-n...


Next, we leave out all columns except published and url. The former corresponds to the time dimension while the latter uniquely identifies a post by its URL. Along the way we get rid of possible duplicates and missing values in the data:

In [7]:
df = df[["published", "url"]].dropna().drop_duplicates()

Next, we need to convert published to the datetime format because by default pandas treats this field as string-valued.

In [8]:
df["published"] = pd.to_datetime(df["published"])

In [9]:
df.sort_values(by=["published"]).head(n=2)

,published,url
50931,1970-01-01 00:00:00.001000+00:00,https://medium.com/iiot
40243,1970-01-01 00:00:00.001000+00:00,https://medium.com/@ikaella/melon-rebranding-b...


Medium’s public release date was August 15, 2012. But, as you can see from the data above, there are at least several rows with much earlier publication dates. They have somehow turned up in our dataset, but they are hardly legitimate ones. We will just trim our time series to keep only those rows that fall onto the period from August 15, 2012 to June 25, 2017:

In [10]:
df = df[(df["published"] > "2012-08-15") & (df["published"] < "2017-06-26")].sort_values(by=["published"])
df.head(n=3)

,published,url
24630,2012-08-15 00:25:03.373000+00:00,https://medium.com/launch-day/jean-attempts-to...
24631,2012-08-15 00:25:29.419000+00:00,https://medium.com/launch-day/dan-and-kristin-...
17811,2012-08-15 00:34:59.502000+00:00,https://medium.com/i-m-h-o/the-world-is-social...


In [11]:
df.tail(n=3)

,published,url
62122,2017-06-25 23:36:01.171000+00:00,https://medium.com/push-the-pace/the-official-...
72471,2017-06-25 23:41:48.295000+00:00,https://medium.com/parti-xyz-developers/%EA%B4...
83283,2017-06-25 23:51:43+00:00,http://www.johanr.com/blog/people-support-dreams


In [12]:
aggr_df = df.groupby("published")[["url"]].count()
aggr_df.columns = ["posts"]

In this practice, we are interested in the number of posts a day. But at this moment all our data is divided into irregular time intervals that are less than a day. This is called a sub-daily time series. To see it, let’s print out the first 3 rows:

In [13]:
aggr_df.head(n=3)

,posts
published,
2012-08-15 00:25:03.373000+00:00,1
2012-08-15 00:25:29.419000+00:00,1
2012-08-15 00:34:59.502000+00:00,1


To fix this, we need to aggregate the post counts by “bins” of a date size. In time series analysis, this process is referred to as resampling. If we reduce the sampling rate of data, it is often called downsampling.

Luckily, pandas has a built-in functionality for this task. We will resample our time index down to 1-day bins:

In [14]:
daily_df = aggr_df.resample("D").apply(sum)
daily_df.head(n=3)

,posts
published,
2012-08-15 00:00:00+00:00,16
2012-08-16 00:00:00+00:00,11
2012-08-17 00:00:00+00:00,4


##  Exploratory visual analysis

In [15]:
from plotly import graph_objs as go
from plotly.offline import init_notebook_mode, iplot, plot
from IPython.display import display, IFrame

# initilise plotly
init_notebook_mode(connected=True)

define a helper function, which will plot our dataframes throughout the article:

```python
common_kw = dict(x=df.index, mode="lines")
```

is basically the same as:

```python
common_kw = {
    "x": df.index,
    "mode": "lines"
}
```
This is commonly used with **Plotly**.

For example, you might later write:

```python
fig.add_scatter(y=df["posts"], **common_kw)
```

The `**common_kw` means:

> "Take the things inside `common_kw` and pass them as arguments."

So this:

```python
fig.add_scatter(y=df["posts"], **common_kw)
```

is essentially like writing:

```python
fig.add_scatter(
    y=df["posts"],
    x=df.index,
    mode="lines"
)
```


In [16]:
def plotly_df(df, title="", width=800, height=500):
    """Visualize all the dataframe columns as line plots."""
    common_kw = dict(x=df.index, mode="lines")
    data = [go.Scatter(y=df[c], name=c, **common_kw) for c in df.columns]
    layout = dict(title=title)
    fig = dict(data=data, layout=layout)

    # in a Jupyter Notebook, the following should work
    iplot(fig, show_link=False)

    # in a Jupyter Book, we save a plot offline and then render it with IFrame
    # plot_path = f"./{title}.html".replace(" ", "_")
    # plot(fig, filename=plot_path, show_link=False, auto_open=False);
    # display(IFrame(plot_path, width=width, height=height))

Let’s try and plot our dataset as is:

In [17]:
plotly_df(daily_df, title="Posts on Medium(daily)")

High-frequency data can be rather difficult to analyze.
To reduce the noise, we will resample the post counts down to weekly bins.
We save our downsampled dataframe in a separate variable because further in this practice we will work only with daily series:

In [18]:
weekly_df = daily_df.resample("W").agg(sum)

Finally we plot the result

In [19]:
plotly_df(weekly_df, title="Posts on Medium (weekly)")

Now, we’re going to omit the first few years of observations, up to 2015. First, they won’t contribute much into the forecast quality in 2017. Second, these first years, having very low number of posts per day, are likely to increase noise in our predictions, as the model would be forced to fit this abnormal historical data along with more relevant and indicative data from the recent years.

In [20]:
daily_df = daily_df.loc[daily_df.index >= "2015-01-01"]
daily_df.head()

,posts
published,
2015-01-01 00:00:00+00:00,8
2015-01-02 00:00:00+00:00,11
2015-01-03 00:00:00+00:00,11
2015-01-04 00:00:00+00:00,8
2015-01-05 00:00:00+00:00,27


from visual analysis we can see that our dataset is non-stationary with a prominent growing trend. It also demonstrates weekly and yearly seasonality and a number of abnormal days in each year.

## Making a forecast

Prophet’s API is very similar to the one you can find in sklearn. First we create a model, then call the method fit, and, finally, make a forecast. The input to the method fit is a DataFrame with two columns:

- ds (datestamp) must be of type date or datetime.

- y is a numeric value we want to predict.

In [22]:
import logging
from prophet import Prophet
logging.getLogger().setLevel(logging.ERROR)

Let’s convert our dataframe to the format required by Prophet:

In [23]:
df = daily_df.reset_index()
df.columns = ["ds", "y"]
# converting timezones (issue https://github.com/facebook/prophet/issues/831)
df["ds"] = df["ds"].dt.tz_convert(None) # removes the timezone information, making the datetime timezone-naive.
df.tail()

,ds,y
902,2017-06-21,422
903,2017-06-22,441
904,2017-06-23,421
905,2017-06-24,277
906,2017-06-25,253


The authors of the library generally advise to make predictions based on at least several months, ideally, more than a year of historical data. Luckily, in our case we have more than a couple of years of data to fit the model.
To measure the quality of our forecast, we need to split our dataset into the historical part, which is the first and biggest slice of our data, and the prediction part, which will be located at the end of the timeline. We will remove the last month from the dataset in order to use it later as a prediction target:

In [24]:
prediction_size = 30
train_df = df[:-prediction_size] # start from the beginning and stop 30 rows before the end
train_df.tail(n=3)

,ds,y
874,2017-05-24,375
875,2017-05-25,298
876,2017-05-26,269
